## Installations

In [ ]:
%pip install matplotlib
from pathlib import Path
from time import perf_counter
import json
import re
import requests
import pandas as pd

from ollama import chat
OLLAMA_MODEL= "llama3.2:3b"
try:
    response = requests.get("http://localhost:11434")
    print("Ollama is running!")
except:
    print("Ollama is NOT running.")
    print("Start Ollama before continuing.")

## Load dataset/put in pandas DF

In [ ]:
#Testing Variables
TEMPERATURE = 0.2

EVALUATION_SCALE_MIN = 1
EVALUATION_SCALE_MAX = 5

In [1]:
import pandas as pd
import kagglehub
import os
import numpy as np # linear algebra
import pandas as pd
from ollama import chat
from IPython.display import display, HTML
from kagglehub import KaggleDatasetAdapter

# Download latest version
path = kagglehub.dataset_download("sharmaabhi04/100k-movies-dataset")

#CSV File
file_path = "100k_Movies_dataset.csv"  # replace with whatever os.listdir() showed you

#load into pandas DF - load CSV file into a pandas DataFrame``
movies_df = pd.read_csv(os.path.join(path, file_path))

/Users/yinglu/Desktop/movie-assistant/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#explore data
print(movies_df.shape)
# movies_df.columns.tolist()
# movies_df.info()
# movies_df.describe()
# movies_df.head()

# RAG

In [9]:
# Load an embedding model
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5270.13it/s]


Embedding model loaded.


In [4]:
# Chunking
print(movies_df.columns.tolist())
# word based chunking

def split_text_into_chunks(
    text,
    chunk_size=30,
    overlap=5
):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

['title', 'id', 'runtime', 'genre', 'ratings', 'director', 'cast', 'Description', 'released_year', 'movie_link']


In [23]:
print(len(movies_df))

100047


In [ ]:
#Chunking 
all_chunks = []
all_metadata = []


for index, row in movies_df.iterrows(): #loop through each row in the DataFrame

    chunks = split_text_into_chunks(#chunk by title row
        movies_df["title"].astype(str).str.cat(sep=" "), 
        chunk_size=100,
        overlap=20
    )

    for chunk_id, chunk in enumerate(chunks): #loop through each title chunk 

        all_chunks.append(chunk) #store chunk

        all_metadata.append({ #append its associated data
            "runtime": index,
            "genre": row["genre"],
            "ratings": row["ratings"],
            "director": row["director"],
            "cast": row["cast"],
            "Description": row["Description"],
            "released_year": row["released_year"],
            "chunk_id": chunk_id
        })
print("Total CSV chunks:", len(all_chunks))

ValueError: too many values to unpack (expected 2)

## Functions

In [51]:
# retrieval function
def retrieve_context(
    question,
    number_of_results=3
):
    question_embedding = embedding_model.encode(
        question
    ).tolist()

    results = collection.query(
    query_embeddings=[question_embedding],
    n_results=number_of_results
    )
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    return documents, metadatas, distances

In [52]:

# function to generate answer
def generate_rag_answer(
    question,
    number_of_results=3
):
    print("user_question:", question)
    
    documents, metadatas, distances = retrieve_context(
        question,
        number_of_results
    )

    context = "\n\n".join(documents)

    prompt = f"""
You are a movie recommendation assistant.

Answer the question using only the context below.

If the answer is not available in the context, say:
"I do not have enough verified information to answer
that question."

Context:
{context}

Question:
{question}

Answer:
"""

    response = chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.2
        }
    )

    answer = response["message"]["content"]

    return {
        "question": question,
        "answer": answer,
        "documents": documents,
        "metadata": metadatas,
        "distances": distances
    }

In [54]:
# ollama
def generate_response(
    prompt,
    temperature=TEMPERATURE
):
    start_time = perf_counter()

    response = chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": temperature
        }
    )

    elapsed_time = perf_counter() - start_time

    return {
        "text": response["message"]["content"],
        "response_time": elapsed_time
    }

## Vectors in ChromaDB 

In [10]:
## ChromaDB
# creating db client
import chromadb

chroma_client = chromadb.Client()

print("ChromaDB client created.")

ChromaDB client created.


In [12]:
# create collection
collection_name = "movie_information"
try:
    chroma_client.delete_collection(
        collection_name
    )
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name
)

print("Collection created:", collection_name)

Collection created: movie_information


In [13]:
# chunk embedding
chunk_embeddings = embedding_model.encode(
    chunks
).tolist()

print(
    len(chunk_embeddings),
    "chunk embeddings created."
)

1 chunk embeddings created.


In [17]:
collection.add(
    ids=chunk_num,   
    documents=all_chunks,
    metadatas=all_metadata,
    embeddings=chunk_embeddings
)
print("Documents stored in ChromaDB.")
print("Collection size:", collection.count())

ValueError: Unequal lengths for fields: ids: 1, metadatas: 100047, embeddings: 1, documents: 100047 in add.

## Semantic Retrival

In [ ]:
question = "Give a list of comedy movies realsed in 2020 with rating 8 or higher."
print(question)
# ques embedding generation
question_embedding = embedding_model.encode(
    question
).tolist()

print("Query embedding created.")    




Give a list of comedy movies realsed in 2020 with rating 8 or higher.
Query embedding created.
{'ids': [[]], 'embeddings': None, 'documents': [[]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[]], 'distances': [[]]}


In [58]:
# top 3 chunks
results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)

print(results)

{'ids': [[]], 'embeddings': None, 'documents': [[]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[]], 'distances': [[]]}


In [56]:
# display documents retrieved
retrieved_documents = results["documents"][0]
retrieved_distances = results["distances"][0]
retrieved_metadata = results["metadatas"][0]

for index, document in enumerate(
    retrieved_documents
):
    print(f"Result {index + 1}")
    print("Document:", document)
    print("Distance:", retrieved_distances[index])
    print("Metadata:", retrieved_metadata[index])
    print("-" * 60)

## Build Rag Prompt

In [47]:
# combine all the context that was retrieved
context = "\n\n".join(
    retrieved_documents
)

print(context)

In [49]:
# grounded prompt creation
prompt = f"""
You are a Movie Recommendation assistant.

Answer the question using only the provided context.

If the context does not contain enough information,
say:

"I do not have enough verified information to answer
that question."

Context:
{context}

Question:
{question}

Answer:
"""

print(prompt)


You are a Movie Recommendation assistant.

Answer the question using only the provided context.

If the context does not contain enough information,
say:

"I do not have enough verified information to answer
that question."

Context:


Question:
Give a list of comedy movies realsed in 2020 with rating 8 or higher.

Answer:



In [50]:
#Rag Answer
response = chat(
    model=OLLAMA_MODEL,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    options={
        "temperature": 0.2
    }
)

answer = response["message"]["content"]

print(answer)

Here are some comedy movies released in 2020 with a rating of 8 or higher:

1. Borat Subsequent Moviefilm (2020) - 7.5/10
2. Palm Springs (2020) - 7.4/10
3. The Lovebirds (2020) - 6.9/10
4. Eurovision Song Contest: The Story of Fire Saga (2020) - 6.8/10

Note that the ratings may vary slightly depending on the source.

If you'd like more recommendations or have any specific preferences (e.g. genre, actor), feel free to ask!


# Tests

## System Testing

In [33]:
system_prompt = """
You are our AI movie recommendation assistant.
Your job is to recommend movies based on user preferences.
You should:
- Ask questions when the user gives unclear preferences.
- Explain why each recommendation matches the user's interests.
- Never invent movies, actors, ratings, or availability.
- Admit when you do not know something.
"""
print("====SYSTEM PROMPT=====")
print(system_prompt)

====SYSTEM PROMPT=====

You are our AI movie recommendation assistant.
Your job is to recommend movies based on user preferences.
You should:
- Ask questions when the user gives unclear preferences.
- Explain why each recommendation matches the user's interests.
- Never invent movies, actors, ratings, or availability.
- Admit when you do not know something.



In [37]:
print("\n===WEAK PROMPTS===\n")
weak_prompts = [
    "Recommend me some movies.",
    "I need a good movie.",
    "What movie should I watch?",
    "Give me something interesting.",
    "Tell me the best movie."
]
print("1. ", weak_prompts[0])
print("2. ", weak_prompts[1])
print("3. ", weak_prompts[2])
print("4. ", weak_prompts[3])
print("5. ", weak_prompts[4])
print("\n===STRONG PROMPTS===\n")

strong_prompts = [
    "Recommend 5 science fiction movies similar to Interstellar. I like space exploration and emotional stories.",

    "I want a family-friendly comedy movie that teenagers and adults can enjoy. Avoid R-rated movies.",

    "I enjoyed Knight and the Seven Kingdoms. Recommend a series with similar themes and storytelling.",

    "Recommend horror movies that focus on suspense instead of gore.",

    "I only have 90 minutes. Recommend highly-rated movies under that runtime."

]
print("1. ", strong_prompts[0])
print("2. ", strong_prompts[1])
print("3. ", strong_prompts[2])
print("4. ", strong_prompts[3])
print("5. ", strong_prompts[4])


===WEAK PROMPTS===

1.  Recommend me some movies.
2.  I need a good movie.
3.  What movie should I watch?
4.  Give me something interesting.
5.  Tell me the best movie.

===STRONG PROMPTS===

1.  Recommend 5 science fiction movies similar to Interstellar. I like space exploration and emotional stories.
2.  I want a family-friendly comedy movie that teenagers and adults can enjoy. Avoid R-rated movies.
3.  I enjoyed Knight and the Seven Kingdoms. Recommend a series with similar themes and storytelling.
4.  Recommend horror movies that focus on suspense instead of gore.
5.  I only have 90 minutes. Recommend highly-rated movies under that runtime.


## Functional Testing

In [ ]:
# testing complete RAG
question = "Recommend 3 scary movies above 4 stars"
result = generate_rag_answer(question, 3)
print("Question:")

print(result["question"])

print("\nAnswer:")
print(result["answer"])

## RAG Testing

In [ ]:
import ollama
import requests
import subprocess
import time

messages=[
    {
        "role":"user",
        "content":"Recommend movies 3 stars and up."
    }
]

# Call ollama.chat with stream=False to get the full response as a dictionary
response = ollama.chat(
    model="llama3.2:3b",
    messages=messages,
    stream=False # Set stream to False to get a dictionary response
)

# Now, 'response' is a dictionary, and you can access its elements
print(response["message"]["content"])

In [ ]:
# Few shot prompting
prompt = """
Movie Based Reccomendations

Example 1

Ex: User asks: I want something funny to cheer me up
Assistant: Here are some great comedy movies:

Paddington 2, Chef, and the Princess Bride


Example 2

User asks: I want something emotional
Assistant: Ok, here are some emotional movies you might like
Manchester by the sea
About Time
Interstellar


Example 3

Recommend based on favorite Movies
Ex. User asks: I loved Spider-Man: No way Home, Avengers: End Game, and
Guardians of the Galaxy
Assistant: You seem to enjoy superhero movies. Based on that, I recommend:
Shang-Chi and the Legend of the Ten Rings
Thor: Ragnarok
The Suicide Squad

"""

response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(response["message"]["content"])

In [ ]:
# System Prompt
response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"system",
            "content":"You are a parent of a young child, what movies would you recommend?"
        },
        {
            "role":"user",
            "content":"Recommend Movies."
        }
    ]
)

print(response["message"]["content"])

In [ ]:
# One-shot prompting
prompt = """
Example

Sentence:
The movie was fantastic.

Sentiment:
Positive

Sentence:
The assignment was confusing.

Sentiment:
"""

response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(response["message"]["content"])

## Prompt Testing

In [ ]:
print("\n===ZERO-SHOT PROMPTS===\n")

zero_shot_prompts = [
    "Recommend a movie for a rainy day.",
    "What's a good movie to watch with my parents?",
    "Suggest a movie similar to Inception.",
    "Give me a movie recommendation for someone who likes slow-burn dramas.",
    "What should I watch if I only have 90 minutes free?"
]

print("1. ", zero_shot_prompts[0])
print("2. ", zero_shot_prompts[1])
print("3. ", zero_shot_prompts[2])
print("4. ", zero_shot_prompts[3])
print("5. ", zero_shot_prompts[4])

In [ ]:
print("\n===ZERO-SHOT RESPONSES===\n")

for i, prompt in enumerate(zero_shot_prompts, start=1):
    response = chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.7
        }
    )
    print(f"{i}. PROMPT: {prompt}")
    print(f"   RESPONSE: {response['message']['content']}")
    print("\n")

## Hallucination Testing

In [ ]:
print("\n===HALLUCINATION PROMPTS===\n")
hallucination_prompts = [
    "Tell me about the movie The Last Ocean Planet starring Chris Evans.",

    "Why did Titanic 2: Return of the Ocean win Best Picture?",

    "Recommend movies directed by Christopher Nolan before 1900.",

    "What superhero movies did Emily Watson Jr. star in?",

    "Is Avatar 5 currently available on Netflix?"
]
print("1. ", hallucination_prompts[0])
print("2. ", hallucination_prompts[1])
print("3. ", hallucination_prompts[2])
print("4. ", hallucination_prompts[3])
print("5. ", hallucination_prompts[4])

In [ ]:
# safe prompt
def create_grounded_prompt(
    question,
    context
):
    return f"""
You are a movie recommendation assistant.

Answer only using the supplied context.

If the context does not contain the answer,
respond exactly with:

"I do not have enough infor to answer this questions"

Do not guess.
Do not invent information.
Do not use outside knowledge.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
# test
unsupported_question = (
    "What is the instructor's private phone number?"
)

safe_prompt = create_grounded_prompt(
    unsupported_question,
    context
)

unsupported_result = generate_response(
    safe_prompt
)

print(unsupported_result["text"])